# Video tweets with no misinformation

A tweet qualifies if either:
1. The top-agreed note classifies NOT_MISLEADING with agree > disagree, OR
2. The top-agreed note classifies MISINFORMED_OR_POTENTIALLY_MISLEADING but has status CURRENTLY_RATED_NOT_HELPFUL (community rejected the correction).

In [5]:
import os
import csv
import glob as globmod
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

# --- Config ---
MIN_DURATION_S = 20
MAX_DURATION_S = 120
MIN_DURATION_MS = MIN_DURATION_S * 1000
MAX_DURATION_MS = MAX_DURATION_S * 1000
RATINGS_CHUNK_SIZE = 5_000_000

CN_DATA_DIR = os.path.join("..", "cn_data")
CN_NOTES_TSVS = sorted(globmod.glob(os.path.join(CN_DATA_DIR, "notes-*.tsv")))
CN_RATINGS_TSVS = sorted(globmod.glob(os.path.join(CN_DATA_DIR, "ratings-*.tsv")))
STATUS_HISTORY_TSV = os.path.join(CN_DATA_DIR, "noteStatusHistory-00000.tsv")
OUTPUT_CSV = "video_no_misinfo.csv"

print(f"Notes files: {len(CN_NOTES_TSVS)}")
print(f"Ratings files: {len(CN_RATINGS_TSVS)}")

Notes files: 2
Ratings files: 8


In [6]:
# --- Supabase connection ---
load_dotenv(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".env.local"))
supabase = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_SERVICE_KEY"])
print("Connected to Supabase")

Connected to Supabase


In [7]:
# --- Load CN notes ---
print("Loading CN notes TSVs...")
notes_cn = pd.concat(
    [pd.read_csv(f, sep="\t", dtype=str, usecols=["noteId", "tweetId", "classification", "summary"])
     for f in CN_NOTES_TSVS],
    ignore_index=True,
)
print(f"  {len(notes_cn)} notes loaded from {len(CN_NOTES_TSVS)} files")
notes_cn.head()

Loading CN notes TSVs...
  2506283 notes loaded from 2 files


,noteId,tweetId,classification,summary
0,1783179305159200982,1783159712986382830,MISINFORMED_OR_POTENTIALLY_MISLEADING,The House failed to pass a border protection l...
1,1783181538789605871,1783171851818021181,MISINFORMED_OR_POTENTIALLY_MISLEADING,The United States has 50 States https://da...
2,1783182562279494134,1783154445682979015,MISINFORMED_OR_POTENTIALLY_MISLEADING,TikTok only mentions “ban” and chooses to igno...
3,1883711635770196070,1883619411774345444,MISINFORMED_OR_POTENTIALLY_MISLEADING,This could be considered a threat https://...
4,1537142913737428992,1377030478167937024,MISINFORMED_OR_POTENTIALLY_MISLEADING,Forbes has a good rundown of the investigation...


In [8]:
# --- Load status history ---
print("Loading status history TSV...")
status_history = pd.read_csv(STATUS_HISTORY_TSV, sep="\t", dtype=str, usecols=["noteId", "currentStatus"])
print(f"  {len(status_history)} status records loaded")
status_history.head()

Loading status history TSV...
  2687409 status records loaded


,noteId,currentStatus
0,1352796878438424576,NEEDS_MORE_RATINGS
1,1353415873227177985,NEEDS_MORE_RATINGS
2,1354586938863443971,NEEDS_MORE_RATINGS
3,1354588003075764229,NEEDS_MORE_RATINGS
4,1354588172659920899,NEEDS_MORE_RATINGS


In [9]:
# --- Aggregate agree/disagree from ratings (chunked for ~37GB) ---
print(f"Aggregating ratings from {len(CN_RATINGS_TSVS)} files...")
agree_totals = pd.Series(dtype="int64")
disagree_totals = pd.Series(dtype="int64")

for ratings_file in CN_RATINGS_TSVS:
    print(f"  Processing {os.path.basename(ratings_file)}...")
    for chunk in pd.read_csv(
        ratings_file,
        sep="\t",
        dtype={"noteId": str, "agree": "Int8", "disagree": "Int8"},
        usecols=["noteId", "agree", "disagree"],
        chunksize=RATINGS_CHUNK_SIZE,
    ):
        chunk_agree = chunk.groupby("noteId")["agree"].sum()
        chunk_disagree = chunk.groupby("noteId")["disagree"].sum()
        agree_totals = agree_totals.add(chunk_agree, fill_value=0)
        disagree_totals = disagree_totals.add(chunk_disagree, fill_value=0)

rating_counts = pd.DataFrame({"agree": agree_totals.astype(int), "disagree": disagree_totals.astype(int)})
rating_counts.index.name = "noteId"
rating_counts = rating_counts.reset_index()
print(f"  {len(rating_counts)} notes with ratings")
rating_counts.head()

Aggregating ratings from 8 files...
  Processing ratings-00000.tsv...
  Processing ratings-00001.tsv...
  Processing ratings-00002.tsv...
  Processing ratings-00003.tsv...
  Processing ratings-00004.tsv...
  Processing ratings-00005.tsv...
  Processing ratings-00006.tsv...
  Processing ratings-00007.tsv...
  2559808 notes with ratings


,noteId,agree,disagree
0,1352796878438424576,4,0
1,1353415873227177985,5,1
2,1354586938863443971,0,0
3,1354590891764293637,1,0
4,1354600130578624514,2,7


In [13]:
# Show first 100 rows of rating_counts without truncation
with pd.option_context('display.max_rows', 100, 'display.max_columns', None, 'display.width', None):
    display(rating_counts.head(100))

,noteId,agree,disagree
0,1352796878438424576,4,0
1,1353415873227177985,5,1
2,1354586938863443971,0,0
3,1354590891764293637,1,0
4,1354600130578624514,2,7
5,1354602688097308676,0,0
6,1354630645385846789,7,0
7,1354635010423328769,1,0
8,1354640847245955073,0,1
9,1354646870681767942,2,1


In [14]:
# --- Join notes with ratings and status ---
notes_with_ratings = notes_cn.merge(rating_counts, on="noteId", how="inner")
notes_with_ratings = notes_with_ratings.merge(status_history, on="noteId", how="left")
print(f"  {len(notes_with_ratings)} notes matched with ratings")
notes_with_ratings.head()

  2268835 notes matched with ratings


,noteId,tweetId,classification,summary,agree,disagree,currentStatus
0,1783179305159200982,1783159712986382830,MISINFORMED_OR_POTENTIALLY_MISLEADING,The House failed to pass a border protection l...,0,0,NEEDS_MORE_RATINGS
1,1783181538789605871,1783171851818021181,MISINFORMED_OR_POTENTIALLY_MISLEADING,The United States has 50 States https://da...,0,0,NEEDS_MORE_RATINGS
2,1783182562279494134,1783154445682979015,MISINFORMED_OR_POTENTIALLY_MISLEADING,TikTok only mentions “ban” and chooses to igno...,0,0,NEEDS_MORE_RATINGS
3,1883711635770196070,1883619411774345444,MISINFORMED_OR_POTENTIALLY_MISLEADING,This could be considered a threat https://...,0,0,CURRENTLY_RATED_NOT_HELPFUL
4,1537142913737428992,1377030478167937024,MISINFORMED_OR_POTENTIALLY_MISLEADING,Forbes has a good rundown of the investigation...,0,0,NEEDS_MORE_RATINGS


In [15]:
# --- Pick highest-agreed note per tweet ---
top_notes = notes_with_ratings.sort_values("agree", ascending=False).drop_duplicates(
    subset="tweetId", keep="first"
)
print(f"  {len(top_notes)} unique tweets after keeping top note per tweet")
top_notes.head()

  1434112 unique tweets after keeping top note per tweet


,noteId,tweetId,classification,summary,agree,disagree,currentStatus
245089,1409148577516040196,1408855547852513289,MISINFORMED_OR_POTENTIALLY_MISLEADING,The governor signed an emergency declaration l...,79,5,CURRENTLY_RATED_HELPFUL
427126,1404429847527624708,1404188155897655299,MISINFORMED_OR_POTENTIALLY_MISLEADING,Apple has not removed Grindr from the App Stor...,74,2,CURRENTLY_RATED_HELPFUL
409210,1400245321750310913,1400150275676160001,MISINFORMED_OR_POTENTIALLY_MISLEADING,Local officials held back aid that the Trump a...,55,25,NEEDS_MORE_RATINGS
1042830,1375058804929335300,1374911222361956359,MISINFORMED_OR_POTENTIALLY_MISLEADING,Amazon has a documented history of labor viola...,53,1,CURRENTLY_RATED_HELPFUL
816354,1405987505900560398,1405980032816713728,MISINFORMED_OR_POTENTIALLY_MISLEADING,Congresswoman Omar became a United States citi...,50,0,CURRENTLY_RATED_HELPFUL


In [ ]:
# --- Filter: tweet qualifies as "no misinformation" ---
is_not_misleading = (
    (top_notes["classification"] == "NOT_MISLEADING")
    & (top_notes["agree"] > top_notes["disagree"])
)
is_rejected_correction = (
    (top_notes["classification"] == "MISINFORMED_OR_POTENTIALLY_MISLEADING")
    & (top_notes["currentStatus"] == "CURRENTLY_RATED_NOT_HELPFUL")
    & (top_notes["agree"] > top_notes["disagree"])
)
no_misinfo = top_notes[is_not_misleading | is_rejected_correction].copy()

count_nm = is_not_misleading.sum()
count_rc = is_rejected_correction.sum()
print(f"  {len(no_misinfo)} tweets with no misinformation ({count_nm} NOT_MISLEADING, {count_rc} rejected corrections)")
no_misinfo.head()

In [ ]:
# --- Probe each tweet with yt-dlp for video metadata (no download) ---
import subprocess
import json

def get_video_duration_ms(tweet_id: str) -> float | None:
    """Use yt-dlp -J to get video duration without downloading. Returns duration in ms or None."""
    url = f"https://x.com/i/web/status/{tweet_id}"
    try:
        result = subprocess.run(
            ["yt-dlp", "-J", url],
            capture_output=True, text=True, timeout=60,
        )
        if result.returncode != 0:
            return None
        meta = json.loads(result.stdout)
        duration = meta.get("duration")
        if duration is not None:
            return duration * 1000
        return None
    except Exception:
        return None

tweet_ids = no_misinfo["tweetId"].tolist()
print(f"Probing {len(tweet_ids)} tweets with yt-dlp...")

durations = {}
for i, tid in enumerate(tweet_ids):
    dur = get_video_duration_ms(tid)
    durations[tid] = dur
    tag = f"{dur/1000:.1f}s" if dur else "no video"
    if (i + 1) % 10 == 0 or dur is not None:
        print(f"  [{i+1}/{len(tweet_ids)}] {tid}: {tag}")

no_misinfo["video_duration_ms"] = no_misinfo["tweetId"].map(durations)
has_video = no_misinfo["video_duration_ms"].notna()
print(f"\n  {has_video.sum()} tweets have video out of {len(tweet_ids)}")

In [ ]:
# --- Filter to short-form video ---
video_notes = no_misinfo[
    no_misinfo["video_duration_ms"].notna()
    & (no_misinfo["video_duration_ms"] >= MIN_DURATION_MS)
    & (no_misinfo["video_duration_ms"] <= MAX_DURATION_MS)
].copy()
print(f"  {len(video_notes)} short-form video tweets ({MIN_DURATION_S}-{MAX_DURATION_S}s)")
video_notes.head()

In [ ]:
# --- Build output and save CSV ---
video_notes["video_duration_s"] = (video_notes["video_duration_ms"] / 1000).round(1)
video_notes["tweet_link"] = "https://x.com/i/web/status/" + video_notes["tweetId"]
video_notes["note_link"] = "https://x.com/i/communitynotes/n/" + video_notes["noteId"]

video_notes = video_notes.sort_values("agree", ascending=False)

output = video_notes[[
    "tweet_link", "note_link", "classification", "currentStatus",
    "summary", "agree", "disagree", "video_duration_s",
]].rename(columns={"summary": "note_text"})

output.to_csv(OUTPUT_CSV, index=False, quoting=csv.QUOTE_ALL)
print(f"Output: {OUTPUT_CSV}")
print(f"Total rows: {len(output)}")
output.head(10)